# Building RAG on my dataset

## Flow: File/ Doc reading

### Step 1: Read all files 

In [1]:
from pathlib import Path

# 1. Define the folder path
folder_path = Path("./sample_docs")

print(f"Checking directory: {folder_path.resolve()}")
print(f"Does folder exist?: {folder_path.exists()}\n")

if folder_path.exists():
    print("Files found in this folder:")
    # Look for ALL files (*), not just .txt
    files = list(folder_path.glob("*"))
    if not files:
        print("[None] The folder is empty.")
file_names_list = []

all_content = []
# 2. Loop through files (e.g., all text files)
for file_path in folder_path.glob("*.txt"):
    if file_path.is_file():  # Ensure it is a file, not a subfolder
        print(f"--- Reading: {file_path.name} ---")
        file_names_list.append(file_path.name)
        # 3. Open and read the file safely
        with open(file_path, mode="r", encoding="utf-8") as file:
                content = file.read()
                all_content.append({
                    "id": file_path.name.replace("../",""), "text": content})

print(f"Files: {len(file_names_list)} \n{file_names_list}")
#all_content

Checking directory: /voc/work/week06/sample_docs
Does folder exist?: True

Files found in this folder:
--- Reading: Week 04 Reference - Role of Mathematics in Machine Learning”.txt ---
--- Reading: Week 03_Transcript Calculus.txt ---
--- Reading: Week 02 Reference Randomized Numerical Linear Algebra for Machine Learning.txt ---
--- Reading: Week 04 Reference - Probability Theory Review for Machine Learning Samuel Ieong.txt ---
--- Reading: Week 03 Reference_The Geometry and Calculus of Losses by Robert C. Williamson and Zac Cranko.txt ---
--- Reading: Week 04_Transcript Probability Theory.txt ---


--- Reading: Week 02_Transcript - Linear Algebra.txt ---
Files: 7 
['Week 04 Reference - Role of Mathematics in Machine Learning”.txt', 'Week 03_Transcript Calculus.txt', 'Week 02 Reference Randomized Numerical Linear Algebra for Machine Learning.txt', 'Week 04 Reference - Probability Theory Review for Machine Learning Samuel Ieong.txt', 'Week 03 Reference_The Geometry and Calculus of Losses by Robert C. Williamson and Zac Cranko.txt', 'Week 04_Transcript Probability Theory.txt', 'Week 02_Transcript - Linear Algebra.txt']


## Chunking (sliding window)

Split each document into overlapping windows. We use the simplest strategy:
**sliding window of 200 characters with 40 characters of overlap.**

This is naive — it will cut mid-sentence, mid-word even. We'll see this fail
in Cell 3.5 (chunk strategy comparison).

In [2]:
def chunk_text(text, size=200, overlap=40):
    """Sliding window over characters."""
    if len(text) <= size:
        return [text]
    chunks, i = [], 0
    while i < len(text):
        end = min(i + size, len(text))
        chunks.append(text[i:end])
        if end == len(text):
            break
        i = end - overlap
    return chunks

# Chunk every document; build a flat list with source pointer
all_chunks = []
for doc in documents:
    for chunk_idx, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "chunk_id":  f"{doc['id']}#{chunk_idx}",
            "source_id": doc["id"],
            "text":      chunk,
        })

print(f"Total chunks from {len(documents)} documents: {len(all_chunks)}\n")
for c in all_chunks[:8]:  # show first 8 to avoid wall of text
    marker = "…" if len(c["text"]) == 200 else " "
    print(f"  {c['chunk_id']:30s} [{len(c['text']):3d} chars] {c['text'][:60]}{marker}")
print(f"  ... and {len(all_chunks) - 8} more chunks")

NameError: name 'documents' is not defined

In [23]:
import os
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

from DemoFiles.wk06_pipeline import (
    CORPUS,
    chunk_documents,
    build_index,
    retrieve,
    ask_rag,
    cost_usd,
    DEFAULT_SYSTEM,
) 

In [24]:
chunks = chunk_documents(all_content, size = 400, overlap = 60)

In [25]:
chunks

[{'chunk_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#0',
  'source_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt',
  'text': '“Role of Mathematics in Machine Learning”\nShweta Lamba1, a), Preeti Saini2, b) and Vinay Kukreja3, c), Bhanu Sharma4, d)\n1,2,3,4( Chitkara University Institute of Engineering and Technology, Chitkara University, Punjab, India)\na)shweta.lamba@chitkara.edu.in\nb)Preeti.saini@chitkara.edu.in\nc) Corresponding author: vinay.kukreja@chitkara.edu.in\nd)Bhanu.sharma@chitkara.edu.in\nAbstract.\nContext: Machin'},
 {'chunk_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#1',
  'source_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt',
  'text': '.in\nd)Bhanu.sharma@chitkara.edu.in\nAbstract.\nContext: Machine learning is forming an algorithm that proposes a model that can learn from data and do predictions. It is used to know why something works and why one model is better 

In [26]:
chunks[0]

{'chunk_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#0',
 'source_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt',
 'text': '“Role of Mathematics in Machine Learning”\nShweta Lamba1, a), Preeti Saini2, b) and Vinay Kukreja3, c), Bhanu Sharma4, d)\n1,2,3,4( Chitkara University Institute of Engineering and Technology, Chitkara University, Punjab, India)\na)shweta.lamba@chitkara.edu.in\nb)Preeti.saini@chitkara.edu.in\nc) Corresponding author: vinay.kukreja@chitkara.edu.in\nd)Bhanu.sharma@chitkara.edu.in\nAbstract.\nContext: Machin'}

In [27]:
index = build_index(chunks)

In [39]:
query = "Why  maths for machine learning"
query = "Calculus what is the role in machine learning?"

In [40]:
retrieved = retrieve(query, index)

In [41]:
retrieved[0]

{'chunk_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#49',
 'source_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt',
 'text': 'tical models based on both calculus and statistics. Calculus can be used to implement learning from patterns. Various combinations of states and control are used by the ML model for analysis [11], [12].\nCONCLUSION Prerequisite of ML is Statistics, Calculus, Linear Algebra, and Probability. Statistics is a branch of mathematics that is related to data. It is used to infer information from a given d',
 'vector': [-0.0254364013671875,
  0.0262908935546875,
  0.01461029052734375,
  -0.0302886962890625,
  0.0300140380859375,
  -0.0018100738525390625,
  -0.01218414306640625,
  0.0155181884765625,
  -0.030517578125,
  0.09429931640625,
  0.003147125244140625,
  -0.032806396484375,
  -0.019622802734375,
  0.0292510986328125,
  0.03033447265625,
  -0.0008935928344726562,
  0.0289154052734375,
  0.017181396484375,
  0.023

In [42]:
print((retrieved[1]['text'], retrieved[1]['chunk_id'], retrieved[1]['score']))

('“Role of Mathematics in Machine Learning”\nShweta Lamba1, a), Preeti Saini2, b) and Vinay Kukreja3, c), Bhanu Sharma4, d)\n1,2,3,4( Chitkara University Institute of Engineering and Technology, Chitkara University, Punjab, India)\na)shweta.lamba@chitkara.edu.in\nb)Preeti.saini@chitkara.edu.in\nc) Corresponding author: vinay.kukreja@chitkara.edu.in\nd)Bhanu.sharma@chitkara.edu.in\nAbstract.\nContext: Machin', 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#0', 0.6242694720274935)


In [44]:
response = ask_rag(query, index)
response

{'question': 'Calculus what is the role in machine learning?',
 'answer': 'Calculus helps to find the direction of change in machine learning. It is used to determine how an unknown variable should be adjusted to make predictions more optimal with minimum error [Week 04 Reference - Role of Mathematics in Machine Learning”.txt#4].',
 'sources': ['Week 04 Reference - Role of Mathematics in Machine Learning”.txt#49',
  'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#0',
  'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#4'],
 'tokens_in': 409,
 'tokens_out': 50,
 'retrieved': [{'chunk_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt#49',
   'source_id': 'Week 04 Reference - Role of Mathematics in Machine Learning”.txt',
   'text': 'tical models based on both calculus and statistics. Calculus can be used to implement learning from patterns. Various combinations of states and control are used by the ML model for analysis [11], [12].\n